# Tuning DNN untuk regresi multi-output logistik

DNN dipilih dari validasi saja; test set tetap untuk evaluasi akhir. Data tetap sintetis seperti generator notebook sumber.


In [1]:
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras import Model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Dense, BatchNormalization, Dropout, Normalization, Input, Add,
    Activation, Layer, Concatenate,
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
for _gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(_gpu, True)
np.random.seed(SEED)
tf.random.set_seed(SEED)
n_samples = 50000

c:\Users\Jase.LAPTOP-UM736EL9\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.0) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [2]:
damage_count = np.random.randint(1, 200, size=n_samples)
affected_kk = np.maximum(1, np.round(damage_count * np.random.uniform(0.9, 1.2, size=n_samples))).astype(int)
total_population = affected_kk * np.random.randint(3, 6, size=n_samples)
disaster_type_enc = np.random.randint(0, 6, size=n_samples)
severity_level = np.random.randint(1, 6, size=n_samples)
emergency_duration = np.random.choice([3, 7, 10, 14], size=n_samples)
vulnerability_idx = np.random.uniform(0.1, 1.0, size=n_samples)

duration_factor = emergency_duration / 10.0
vuln_factor = 0.9 + (vulnerability_idx * 0.2)

beras_kg = np.round(affected_kk * 5.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples), 1)
minyak_liter = np.round(affected_kk * 1.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples), 1)
gula_kg = np.round(affected_kk * 1.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples), 1)
indomie_pcs = np.round(affected_kk * 5.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples)).astype(int)
roma_sari_gandum_pack = np.round(affected_kk * 1.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples)).astype(int)
roma_malkist_abon_pack = np.round(affected_kk * 1.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples)).astype(int)
roma_kelapa_pack = np.round(affected_kk * 1.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples)).astype(int)
roma_marie_susu_pack = np.round(affected_kk * 1.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples)).astype(int)
sarden_pcs = np.round(affected_kk * 1.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples)).astype(int)
kornet_pcs = np.round(affected_kk * 1.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples)).astype(int)
susu_full_cream_pcs = np.round(affected_kk * 1.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples)).astype(int)
susu_dancow_box = np.round(affected_kk * 1.0 * duration_factor * vuln_factor * np.random.uniform(0.97, 1.03, size=n_samples)).astype(int)

matras_pcs = np.maximum(1, np.round(affected_kk * np.random.uniform(0.95, 1.05, size=n_samples))).astype(int)
kasur_lipat_pcs = np.maximum(1, np.round(affected_kk * np.random.uniform(0.95, 1.05, size=n_samples))).astype(int)
kompor_set = np.maximum(1, np.round(affected_kk * np.random.uniform(0.95, 1.05, size=n_samples))).astype(int)
karpet_plastik_pcs = np.maximum(1, np.round(affected_kk * np.random.uniform(0.95, 1.05, size=n_samples))).astype(int)
kipas_angin_pcs = np.maximum(1, np.round(affected_kk * 0.7 * (0.8 + 0.1 * severity_level) * np.random.uniform(0.95, 1.05, size=n_samples))).astype(int)

In [3]:
x = pd.DataFrame({
    "damage_count": damage_count,
    "affected_kk": affected_kk,
    "total_population": total_population,
    "disaster_type_enc": disaster_type_enc,
    "severity_level": severity_level,
    "emergency_duration": emergency_duration,
    "vulnerability_idx": vulnerability_idx
})

y = pd.DataFrame({
    "beras_kg": beras_kg,
    "minyak_liter": minyak_liter,
    "gula_kg": gula_kg,
    "indomie_pcs": indomie_pcs,
    "roma_sari_gandum_pack": roma_sari_gandum_pack,
    "roma_malkist_abon_pack": roma_malkist_abon_pack,
    "roma_kelapa_pack": roma_kelapa_pack,
    "roma_marie_susu_pack": roma_marie_susu_pack,
    "sarden_pcs": sarden_pcs,
    "kornet_pcs": kornet_pcs,
    "susu_full_cream_pcs": susu_full_cream_pcs,
    "susu_dancow_box": susu_dancow_box,
    "matras_pcs": matras_pcs,
    "kasur_lipat_pcs": kasur_lipat_pcs,
    "kompor_set": kompor_set,
    "karpet_plastik_pcs": karpet_plastik_pcs,
    "kipas_angin_pcs": kipas_angin_pcs
})

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=SEED)
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=SEED)
print(f"Fit {len(X_fit)} | validation {len(X_val)} | test {len(X_test)}")
x_fit = X_fit.to_numpy(dtype=np.float32)
x_val = X_val.to_numpy(dtype=np.float32)
x_test = X_test.to_numpy(dtype=np.float32)
y_fit_np = y_fit.to_numpy(dtype=np.float32)
y_val_np = y_val.to_numpy(dtype=np.float32)
y_test_np = y_test.to_numpy(dtype=np.float32)

# Standardisasi target per item agar target berskala besar tidak mendominasi MSE.
y_mean = y_fit_np.mean(axis=0)
y_scale = y_fit_np.std(axis=0)
y_scale = np.where(y_scale < 1e-8, 1.0, y_scale)
y_fit_scaled = (y_fit_np - y_mean) / y_scale
y_val_scaled = (y_val_np - y_mean) / y_scale

def fit_keras(model, epochs=100):
    model.compile(optimizer=Adam(learning_rate=0.002), loss="mse", metrics=["mae"])
    history = model.fit(
        x_fit, y_fit_scaled, validation_data=(x_val, y_val_scaled),
        epochs=epochs, batch_size=128, verbose=1,
        callbacks=[EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)],
    )
    pred_scaled = model.predict(x_test, verbose=0)
    # Kembalikan ke satuan target asli sebelum menghitung MAE/RMSE/R?.
    pred_original_units = pred_scaled * y_scale + y_mean
    return pred_original_units, history

def make_normalizer():
    return Normalization(mean=x_fit.mean(axis=0), variance=x_fit.var(axis=0))

Fit 32000 | validation 8000 | test 10000


In [5]:
# Tune DNN candidates using validation macro-MAE (original target units), matching the per-item MAE comparison.
# The test set is evaluated only once, after the best validation candidate is chosen.
dnn_configs = [
    {"widths": [256, 128, 64, 32], "dropout": 0.30, "batch_norm": True,  "learning_rate": 0.002},  # original
    {"widths": [256, 128, 64],     "dropout": 0.10, "batch_norm": True,  "learning_rate": 0.001},
    {"widths": [256, 128, 64],     "dropout": 0.00, "batch_norm": False, "learning_rate": 0.001},
    {"widths": [128, 64, 32],      "dropout": 0.00, "batch_norm": False, "learning_rate": 0.001},
    {"widths": [256, 128, 64, 32], "dropout": 0.00, "batch_norm": False, "learning_rate": 0.0005},
    {"widths": [512, 256, 128, 64], "dropout": 0.05, "batch_norm": True,  "learning_rate": 0.001},
    {"widths": [256, 128, 64], "dropout": 0.00, "batch_norm": False, "learning_rate": 0.001,
     "per_target_heads": True, "head_width": 16},
    {"widths": [256, 128, 64], "dropout": 0.00, "batch_norm": False, "learning_rate": 0.001,
     "loss": "mae"},
]

def build_dnn_candidate(config):
    inputs = Input(shape=(x_fit.shape[1],), name="tabular_features")
    x = make_normalizer()(inputs)
    for layer_index, width in enumerate(config["widths"]):
        x = Dense(width, activation="relu", name=f"shared_dense_{layer_index + 1}")(x)
        if config["batch_norm"]:
            x = BatchNormalization()(x)
        if config["dropout"] > 0:
            x = Dropout(config["dropout"])(x)
    if config.get("per_target_heads", False):
        # Shared representation with a small specialized head for each output.
        outputs = []
        for target_index, target_name in enumerate(y.columns):
            head = Dense(config.get("head_width", 16), activation="relu",
                         name=f"head_{target_index}_{target_name}")(x)
            outputs.append(Dense(1, name=f"output_{target_index}_{target_name}")(head))
        outputs = Concatenate(name="multi_output")(outputs)
    else:
        outputs = Dense(y_fit_np.shape[1], activation="linear", name="outputs")(x)
    return Model(inputs, outputs, name="DNN_candidate")

# Optimalkan kandidat tepat pada macro-MAE satuan asli yang dipakai untuk memilih model.
# Karena output dilatih dalam skala standar, bobot y_scale mengubah MAE scaled menjadi
# MAE satuan asli per target (lalu dirata-ratakan).
class ValidationMacroMAECheckpoint(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best = float("inf")
        self.best_weights = None
        self.best_epoch = 0
        self.wait = 0

    def on_epoch_end(self, epoch, logs=None):
        pred_scaled = self.model.predict(x_val, batch_size=1024, verbose=0)
        pred_original = pred_scaled * y_scale + y_mean
        score = float(mean_absolute_error(
            y_val_np, pred_original, multioutput="raw_values").mean())
        if logs is not None:
            logs["val_macro_mae_original"] = score
        if score < self.best - 1e-6:
            self.best = score
            self.best_epoch = epoch + 1
            self.best_weights = [w.copy() for w in self.model.get_weights()]
            self.wait = 0
        else:
            self.wait += 1
            if self.wait >= 12:
                self.model.stop_training = True


def macro_mae_original_loss(y_true, y_pred):
    return tf.reduce_mean(tf.abs(y_true - y_pred) * tf.constant(y_scale, dtype=tf.float32))


trial_rows = []
best_dnn_val_mae = float("inf")
best_dnn_config = None
best_dnn_weights = None
for trial, config in enumerate(dnn_configs, start=1):
    tf.random.set_seed(SEED)
    candidate = build_dnn_candidate(config)
    candidate.compile(
        optimizer=Adam(learning_rate=config["learning_rate"]),
        loss=macro_mae_original_loss, metrics=["mae"])
    checkpoint = ValidationMacroMAECheckpoint()
    trial_history = candidate.fit(
        x_fit, y_fit_scaled,
        validation_data=(x_val, y_val_scaled),
        epochs=150, batch_size=128, verbose=1,
        callbacks=[checkpoint],
    )
    val_mae_macro = checkpoint.best
    trial_rows.append({"trial": trial, **config, "best_epoch": checkpoint.best_epoch,
                       "validation_MAE_macro": val_mae_macro})
    print(f"Trial {trial}: validation macro-MAE={val_mae_macro:.4f}; config={config}")
    if val_mae_macro < best_dnn_val_mae:
        best_dnn_val_mae = val_mae_macro
        best_dnn_config = config.copy()
        best_dnn_weights = checkpoint.best_weights
    del candidate, trial_history, checkpoint
    tf.keras.backend.clear_session()
    gc.collect()

dnn_tuning_results = pd.DataFrame(trial_rows).sort_values("validation_MAE_macro")
print("Best DNN config:", best_dnn_config)
dnn_tuning_results

# Rebuild best architecture, restore its validation-selected weights, then touch test once.
dnn = build_dnn_candidate(best_dnn_config)
dnn.compile(optimizer=Adam(learning_rate=best_dnn_config["learning_rate"]),
             loss=macro_mae_original_loss, metrics=["mae"])
dnn.set_weights(best_dnn_weights)
pred_dnn_scaled = dnn.predict(x_test, verbose=0)
pred_dnn = pred_dnn_scaled * y_scale + y_mean


Epoch 1/150
250/250 [==============================] - 4s 7ms/step - loss: 42.3763 - mae: 0.4387 - val_loss: 28.1373 - val_mae: 0.2777 - val_macro_mae_original: 28.1374
Epoch 2/150
250/250 [==============================] - 1s 6ms/step - loss: 21.0435 - mae: 0.2165 - val_loss: 8.0820 - val_mae: 0.0853 - val_macro_mae_original: 8.0820
Epoch 3/150
250/250 [==============================] - 1s 5ms/step - loss: 19.4243 - mae: 0.2000 - val_loss: 7.8079 - val_mae: 0.0806 - val_macro_mae_original: 7.8079
Epoch 4/150
250/250 [==============================] - 1s 5ms/step - loss: 19.0911 - mae: 0.1963 - val_loss: 5.5517 - val_mae: 0.0622 - val_macro_mae_original: 5.5517
Epoch 5/150
250/250 [==============================] - 1s 5ms/step - loss: 18.1498 - mae: 0.1875 - val_loss: 7.4505 - val_mae: 0.0805 - val_macro_mae_original: 7.4505
Epoch 6/150
250/250 [==============================] - 1s 4ms/step - loss: 18.3468 - mae: 0.1888 - val_loss: 6.0668 - val_mae: 0.0703 - val_macro_mae_original: 6.0

In [6]:
# MLP lebih kecil sebagai baseline deep learning.
mlp = Sequential([
    Input(shape=(x_fit.shape[1],)), make_normalizer(),
    Dense(64, activation="relu"), Dense(32, activation="relu"),
    Dense(y_fit_np.shape[1], activation="linear"),
], name="MLP_sederhana")
pred_mlp, history_mlp = fit_keras(mlp)

Epoch 1/100
250/250 [==============================] - 1s 4ms/step - loss: 0.1193 - mae: 0.1900 - val_loss: 0.0071 - val_mae: 0.0654
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 0.0051 - mae: 0.0554 - val_loss: 0.0041 - val_mae: 0.0497
Epoch 3/100
250/250 [==============================] - 0s 2ms/step - loss: 0.0035 - mae: 0.0455 - val_loss: 0.0030 - val_mae: 0.0417
Epoch 4/100
250/250 [==============================] - 1s 2ms/step - loss: 0.0027 - mae: 0.0396 - val_loss: 0.0024 - val_mae: 0.0368
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 0.0023 - mae: 0.0359 - val_loss: 0.0022 - val_mae: 0.0344
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 0.0021 - mae: 0.0336 - val_loss: 0.0020 - val_mae: 0.0329
Epoch 7/100
250/250 [==============================] - 1s 2ms/step - loss: 0.0020 - mae: 0.0321 - val_loss: 0.0019 - val_mae: 0.0312
Epoch 8/100
250/250 [==============================] - 1s 2ms/step - 

In [7]:
# Deep & Cross Network (DCN) untuk interaksi fitur tabular, implementasi Keras.
class CrossLayer(Layer):
    def build(self, input_shape):
        feature_dim = int(input_shape[0][-1])
        self.kernel = self.add_weight(
            name="kernel", shape=(feature_dim, 1), initializer="glorot_uniform", trainable=True)
        self.bias = self.add_weight(
            name="bias", shape=(feature_dim,), initializer="zeros", trainable=True)
        super().build(input_shape)

    def call(self, inputs):
        x0, x = inputs
        scalar = tf.matmul(x, self.kernel)
        return x0 * scalar + self.bias + x

    def get_config(self):
        return super().get_config()

inputs = Input(shape=(x_fit.shape[1],), name="tabular_features")
x0 = make_normalizer()(inputs)
cross = x0
for layer_index in range(3):
    cross = CrossLayer(name=f"cross_layer_{layer_index + 1}")([x0, cross])
deep = Dense(128, activation="relu")(x0)
deep = Dense(64, activation="relu")(deep)
combined = Concatenate()([cross, deep])
outputs = Dense(y_fit_np.shape[1], activation="linear")(combined)
dcn = Model(inputs, outputs, name="Deep_and_Cross_Network")
pred_dcn, history_dcn = fit_keras(dcn)


Epoch 1/100
250/250 [==============================] - 1s 4ms/step - loss: 0.5409 - mae: 0.3412 - val_loss: 0.0400 - val_mae: 0.1337
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 0.0245 - mae: 0.1063 - val_loss: 0.0153 - val_mae: 0.0889
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 0.0119 - mae: 0.0784 - val_loss: 0.0088 - val_mae: 0.0687
Epoch 4/100
250/250 [==============================] - 1s 3ms/step - loss: 0.0077 - mae: 0.0645 - val_loss: 0.0062 - val_mae: 0.0586
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 0.0055 - mae: 0.0555 - val_loss: 0.0048 - val_mae: 0.0521
Epoch 6/100
250/250 [==============================] - 1s 3ms/step - loss: 0.0043 - mae: 0.0492 - val_loss: 0.0038 - val_mae: 0.0462
Epoch 7/100
250/250 [==============================] - 1s 2ms/step - loss: 0.0036 - mae: 0.0447 - val_loss: 0.0033 - val_mae: 0.0430
Epoch 8/100
250/250 [==============================] - 1s 3ms/step - 

In [14]:
def calculate_metrics(y_true, y_pred):
    mae_each = mean_absolute_error(y_true, y_pred, multioutput="raw_values")
    rmse_each = np.sqrt(mean_squared_error(y_true, y_pred, multioutput="raw_values"))
    r2_each = r2_score(y_true, y_pred, multioutput="raw_values")
    return {
        "MAE_macro": float(np.mean(mae_each)),
        "RMSE_macro": float(np.mean(rmse_each)),
        "R2_macro": float(np.mean(r2_each)),
        "MAE_weighted_all_values": float(mean_absolute_error(y_true, y_pred)),
        "RMSE_weighted_all_values": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }

predictions = {
    "ANN/MLP": pred_mlp,
    "DCN": pred_dcn,
    "DNN": pred_dnn,
}
summary = pd.DataFrame({name: calculate_metrics(y_test_np, pred) for name, pred in predictions.items()}).T
summary.sort_values("RMSE_macro", ascending=False)

,MAE_macro,RMSE_macro,R2_macro,MAE_weighted_all_values,RMSE_weighted_all_values
ANN/MLP,2.534867,3.518331,0.998338,2.534867,4.377394
DCN,2.478088,3.467971,0.998376,2.478088,4.334905
DNN,2.449849,3.454533,0.998379,2.449849,4.302343


In [13]:
per_item = pd.DataFrame(index=y.columns)
for name, pred in predictions.items():
    per_item[f"{name} MAE"] = mean_absolute_error(y_test_np, pred, multioutput="raw_values")
per_item

,ANN/MLP MAE,DCN MAE,DNN MAE
beras_kg,7.328990,7.420708,7.079399
minyak_liter,1.501635,1.417503,1.404204
gula_kg,1.543876,1.462499,1.452679
indomie_pcs,7.297799,7.026297,7.088434
roma_sari_gandum_pack,1.497350,1.442436,1.463419
roma_malkist_abon_pack,1.515660,1.445762,1.455678
roma_kelapa_pack,1.616574,1.504723,1.445179
roma_marie_susu_pack,1.583813,1.449636,1.445454
sarden_pcs,1.531147,1.509162,1.440508
kornet_pcs,1.539763,1.440537,1.445496
